In [2]:
# run for only first time

# !git clone https://github.com/thuanz123/realfill.git

In [2]:
# run for only first time
# %cd realfill
# %ls

In [3]:
# !pip install -r /content/realfill/requirements.txt
!pip install -r /userhome/cs/yeunggs/APAI3010/APAI3010/requirements.txt

  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [4]:
from accelerate.utils import write_basic_config
write_basic_config()

/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration already exists at /userhome/cs/yeunggs/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [5]:
%pip install "numpy<2"
import numpy
numpy.__version__

Note: you may need to restart the kernel to use updated packages.


'2.2.6'

In [1]:
import numpy
numpy.__version__

'1.26.4'

In [6]:
%pip install diffusers transformers accelerate scipy safetensors 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.0.1+cu117
True


In [1]:
import torch
import os
from diffusers import StableDiffusionInpaintPipeline

# 1. Define the official Model ID
model_id = "sd2-community/stable-diffusion-2-inpainting"

# 2. Check for device (NVIDIA CUDA for the farm, MPS for your Mac)
if torch.cuda.is_available():
    current_device = "cuda"
    current_dtype = torch.float16 # Farm GPUs handle float16 for speed
elif torch.backends.mps.is_available():
    current_device = "mps"
    current_dtype = torch.float32 # Mac is more stable with float32
else:
    current_device = "cpu"
    current_dtype = torch.float32

print(f"Loading pipeline onto: {current_device}")

# 3. Load the pipeline (It will download once and then cache automatically)
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id,
    torch_dtype=current_dtype,
    use_safetensors=True
)

pipe = pipe.to(current_device)

/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pipeline onto: cuda


/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading pipeline components...: 100%|█████████████████████████████████████████████████| 6/6 [00:01<00:00,  4.59it/s]


In [ ]:
# This is used to make sure we use GPU to train
#!accelerate config default

Configuration already exists at /userhome/cs/yeunggs/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


In [3]:
#clean up the cache
import torch
import gc

# Run garbage collection to remove unreferenced variables
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

# Check how much memory is currently allocated
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 0.00 GB
Reserved:  0.00 GB


In [4]:
%cd /userhome/cs/yeunggs/APAI3010/APAI3010/
# from huggingface_hub import login

# base brain: stable diffusion 2.0 inpainting
%env MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
# reference image
%env TRAIN_DIR=realfill_data_release_full/Qualitative/Luminance_Test/3
%env OUTPUT_DIR=Reproduced_results/L3/model

# !accelerate launch train_realfill.py \
!accelerate launch train_realfill.py \
  --pretrained_model_name_or_path=$MODEL_NAME \
  --train_data_dir=$TRAIN_DIR \
  --output_dir=$OUTPUT_DIR \
  --mixed_precision=fp16 \
  --resolution=512 \
  --train_batch_size=16 \
  --gradient_accumulation_steps=1 \
  --unet_learning_rate=2e-4 \
  --text_encoder_learning_rate=4e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=100 \
  --max_train_steps=2000 \
  --lora_rank=8 \
  --lora_dropout=0.1 \
  --lora_alpha=16

  #   # --train_batch_size=16 \

/userhome/cs/yeunggs/APAI3010/APAI3010
env: MODEL_NAME=sd2-community/stable-diffusion-2-inpainting
env: TRAIN_DIR=realfill_data_release_full/Qualitative/Luminance_Test/3
env: OUTPUT_DIR=Reproduced_results/L3/model


/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user fe

In [1]:
%env MODEL_PATH=Reproduced_results/L3/model
%env VAL_IMG=realfill_data_release_full/Qualitative/Luminance_Test/3/target/target.png
%env VAL_MASK=realfill_data_release_full/Qualitative/Luminance_Test/3/target/mask.png
%env OUTPUT_IMG_DIR=Reproduced_results/L3/results

!accelerate launch infer_new.py \
    --model_path=$MODEL_PATH \
    --validation_image=$VAL_IMG \
    --validation_mask=$VAL_MASK \
    --output_dir=$OUTPUT_IMG_DIR \
    --resolution=512

env: MODEL_PATH=Reproduced_results/L3/model
env: VAL_IMG=realfill_data_release_full/Qualitative/Luminance_Test/3/target/target.png
env: VAL_MASK=realfill_data_release_full/Qualitative/Luminance_Test/3/target/mask.png
env: OUTPUT_IMG_DIR=Reproduced_results/L3/results
Using device: cuda
/userhome/cs/yeunggs/APAI3010/APAI3010/.conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|█████████████████████████████████████████| 200/200 [00:14<00:00, 13.59it/s]


In [11]:
!tar -czvf flowerwoman_results.tar.gz flowerwoman-results


tar: flowerwoman-results: Cannot stat: No such file or directory
tar: Exiting with failure status due to previous errors


In [ ]:
# Evaluation function for flowerwoman
import os
import benchmark_suite
import importlib
importlib.reload(benchmark_suite)
from benchmark_suite import RealFillBench

# Assuming your save function is in save_benchmarks.py
import save_benchmarks 
from save_benchmarks import save_to_live_benchmarks

batch = False # Set to True when you are ready for the overnight run
evaluator = RealFillBench(device="cuda")

# --- 1. SINGLE SAMPLE MODE ---
if not batch:
    sample = "1"
    category = "Luminance"
    print(f"--- Running Sanity Check for {sample} ---")
    
    for i in range(16): # Seed loop
        gen_path = f"./Reproduced_results/{category[0]}{sample}/results/{i}.png"
        target_path = f"./realfill_data_release_full/Qualitative/{category}_Test/{sample}/target/target.png"
        ref_folder = f"./realfill_data_release_full/Qualitative/{category}_Test/{sample}/ref"
        
        if os.path.exists(gen_path) and os.path.exists(target_path):
            # Identity scoring across all references
            clip_scores, dino_scores = [], []

            if os.path.exists(ref_folder):
                ref_files = [f for f in os.listdir(ref_folder) if f.lower().endswith(('.png', '.jpg'))]
                for rf in ref_files:
                    rp = os.path.join(ref_folder, rf)
                    clip_scores.append(evaluator.calculate_clip_score(gen_path, rp))
                    dino_scores.append(evaluator.calculate_dino(gen_path, rp))
            
            # Pixel/Perceptual scoring
            results = evaluator.calculate_all(gen_path, target_path)
            
            # Merge identity averages
            if clip_scores:
                results['clip_id'] = sum(clip_scores) / len(clip_scores)
                results['dino_id'] = sum(dino_scores) / len(dino_scores)
            else: #Debug
                print("WARNING: No clip scores were calculated. clip_id and dino_id will be missing.")
            
            
            save_to_live_benchmarks(category, sample, i, results)
            print(f"Seed {i} Results:", results)

# --- 2. BATCH MODE ---
if batch:
    categories = ["Luminance", "Geometry", "Details", "Subject"]
    print("--- Starting Full Batch Evaluation ---")

    for cat in categories:
        for s_idx in range(4): # Loop through samples 0-3 in each category
            for j in range(16): # Loop through seeds 0-15
                # PATHS - Update these to match your actual cluster structure
                gen_path = f"./Reproduced_results/{cat[0]}{s_idx}/results/{j}.png"
                target_path = f"./realfill_data_release_full/Qualitative/{cat}_Test/{s_idx}/target/target.png"
                ref_folder = f"./realfill_data_release_full/Qualitative/{cat}_Test/{s_idx}/input"
                
                if os.path.exists(gen_path) and os.path.exists(target_path):
                    clip_scores, dino_scores = [], []
                    
                    if os.path.exists(ref_folder):
                        ref_files = [f for f in os.listdir(ref_folder) if f.lower().endswith(('.png', '.jpg'))]
                        for rf in ref_files:
                            rp = os.path.join(ref_folder, rf)
                            clip_scores.append(evaluator.calculate_clip_score(gen_path, rp))
                            dino_scores.append(evaluator.calculate_dino(gen_path, rp))
                    
                    results = evaluator.calculate_all(gen_path, target_path)
                    
                    if clip_scores:
                        results['clip_id'] = sum(clip_scores) / len(clip_scores)
                        results['dino_id'] = sum(dino_scores) / len(dino_scores)
                    
                    save_to_live_benchmarks(cat, f"sample_{s_idx}", j, results)
                    print(f"Done: {cat} | Sample {s_idx} | Seed {j}")



Using cached ./models


Using cache found in ./models/facebookresearch_dino_main
Using cache found in ./models/facebookresearch_dino_main


--- Running Sanity Check for 1 ---
Seed 0 Results: {'psnr': 4.548200607299805, 'ssim': 0.3776295781135559, 'lpips': 0.490253210067749, 'dreamsim': 0.3396962881088257, 'clip_id': 0.7586479902267456, 'dino_id': 0.842150616645813}
Seed 1 Results: {'psnr': 5.135563850402832, 'ssim': 0.33888718485832214, 'lpips': 0.49412888288497925, 'dreamsim': 0.30943286418914795, 'clip_id': 0.7604099869728088, 'dino_id': 0.7926730036735534}
Seed 2 Results: {'psnr': 5.057466983795166, 'ssim': 0.3829871416091919, 'lpips': 0.49872761964797974, 'dreamsim': 0.3468165397644043, 'clip_id': 0.7911893606185914, 'dino_id': 0.7955939173698425}
Seed 3 Results: {'psnr': 5.5872111320495605, 'ssim': 0.32852065563201904, 'lpips': 0.500399112701416, 'dreamsim': 0.2948334217071533, 'clip_id': 0.6876836538314819, 'dino_id': 0.7643303155899048}
Seed 4 Results: {'psnr': 6.095787048339844, 'ssim': 0.4360228478908539, 'lpips': 0.478017657995224, 'dreamsim': 0.28319859504699707, 'clip_id': 0.7496509194374085, 'dino_id': 0.79253